# SpatialEffect2 vs SpatialEffect: Ferraro et al. (2015, sf)

Goal: compare core outputs from `SpatialEffect2` and `SpatialEffect` step by step.


In [11]:
import os, glob, hashlib, json, subprocess, shutil
from pathlib import Path

BASE = Path('.')
BASELINE = BASE / 'baseline'  # SpatialEffect
NEW = BASE / 'new'  # SpatialEffect2
print('BASE (relative):', BASE.resolve())
print('SpatialEffect folder exists:', BASELINE.exists())
print('SpatialEffect2 folder exists:', NEW.exists())

RSCRIPT = shutil.which('Rscript')
if RSCRIPT is None:
    for c in [
        '/Library/Frameworks/R.framework/Versions/Current/Resources/bin/Rscript',
        '/Library/Frameworks/R.framework/Versions/4.4-arm64/Resources/bin/Rscript',
        '/usr/local/bin/Rscript',
        '/opt/homebrew/bin/Rscript',
    ]:
        if os.path.exists(c):
            RSCRIPT = c
            break
if RSCRIPT is None:
    raise FileNotFoundError('Rscript not found in PATH or standard locations.')
print('Using Rscript:', RSCRIPT)



BASE (relative): /Users/zheqiao/Documents/cursor_projects/SpatialEffectPackage/WangSamiiChangAronow_2024_replication/application/graphs/SpatialEffect2_Ferraro_etal_2015_sf
SpatialEffect folder exists: True
SpatialEffect2 folder exists: True
Using Rscript: /usr/local/bin/Rscript


## 1) Load core objects from RData (AME, Conley CI, Per CI)


In [12]:
def load_rdata_payload(rdata_path):
    r_path = str(rdata_path).replace('\\', '/').replace('"', '\\"')
    r_expr = '\n'.join([
        'suppressWarnings(suppressPackageStartupMessages(library(jsonlite)))',
        'e <- new.env()',
        f'load("{r_path}", envir = e)',
        'x <- e$result.list',
        'ame <- if (!is.null(x$AME_est)) x$AME_est else x$AMR_est',
        'sm <- if (!is.null(x$AME_est_smoothed)) x$AME_est_smoothed else x$AMR_est_smoothed',
        'out <- list(AME_est = ame, AME_smoothed = sm, Conley_CI = x$Conley.CI, Per_CI = x$Per.CI)',
        "cat(toJSON(out, digits = 16, dataframe = 'columns', auto_unbox = TRUE))"
    ])
    proc = subprocess.run([RSCRIPT, '-e', r_expr], capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"Rscript failed for {rdata_path}\nSTDERR:\n{proc.stderr}\nSTDOUT:\n{proc.stdout}")
    return json.loads(proc.stdout.strip())

spatialeffect_payload = load_rdata_payload(BASELINE / 'Ferraro_etal_2015_sf.RData')
spatialeffect2_payload = load_rdata_payload(NEW / 'Ferraro_etal_2015_sf.RData')
print('Loaded SpatialEffect/SpatialEffect2 payload keys:', spatialeffect_payload.keys())



Loaded SpatialEffect/SpatialEffect2 payload keys: dict_keys(['AME_est', 'AME_smoothed', 'Conley_CI', 'Per_CI'])


## 2) Core numeric differences (max absolute difference)


In [13]:
def max_abs_diff_vec(a, b):
    vals = [abs(float(x) - float(y)) for x, y in zip(a, b)]
    return max(vals) if vals else None


def get_series(payload, key):
    obj = payload.get(key)
    if obj is None:
        return None, None

    # Legacy JSON can be either dict-like columns or list of [lower, upper].
    if isinstance(obj, dict):
        lower = obj.get('1')
        upper = obj.get('2')
        return lower, upper

    if isinstance(obj, list):
        lower = [row[0] for row in obj] if obj else []
        upper = [row[1] for row in obj] if obj else []
        return lower, upper

    return None, None


b_ame = spatialeffect_payload['AME_est']['taud_est']
n_ame = spatialeffect2_payload['AME_est']['taud_est']
print('max |AME_est diff|:', max_abs_diff_vec(b_ame, n_ame))

b_sm_l, b_sm_u = get_series(spatialeffect_payload, 'AME_smoothed')
n_sm_l, n_sm_u = get_series(spatialeffect2_payload, 'AME_smoothed')
if b_sm_l is not None and n_sm_l is not None:
    print('max |AME_smoothed lower diff|:', max_abs_diff_vec(b_sm_l, n_sm_l))
    print('max |AME_smoothed upper diff|:', max_abs_diff_vec(b_sm_u, n_sm_u))

b_c1, b_c2 = get_series(spatialeffect_payload, 'Conley_CI')
n_c1, n_c2 = get_series(spatialeffect2_payload, 'Conley_CI')
if b_c1 is not None and n_c1 is not None:
    print('max |Conley CI lower diff|:', max_abs_diff_vec(b_c1, n_c1))
    print('max |Conley CI upper diff|:', max_abs_diff_vec(b_c2, n_c2))

b_p1, b_p2 = get_series(spatialeffect_payload, 'Per_CI')
n_p1, n_p2 = get_series(spatialeffect2_payload, 'Per_CI')
if b_p1 is not None and n_p1 is not None:
    print('max |Per CI lower diff|:', max_abs_diff_vec(b_p1, n_p1))
    print('max |Per CI upper diff|:', max_abs_diff_vec(b_p2, n_p2))



max |AME_est diff|: 0.0
max |AME_smoothed lower diff|: 0.0
max |AME_smoothed upper diff|: 0.0
max |Conley CI lower diff|: 0.0
max |Conley CI upper diff|: 0.0
max |Per CI lower diff|: 0.0
max |Per CI upper diff|: 0.0


## 3) Row-wise check for the first 10 distance points


In [14]:
d = spatialeffect_payload['AME_est']['d']
for i in range(min(10, len(d))):
    print({
        'd': d[i],
        'AME_SpatialEffect': spatialeffect_payload['AME_est']['taud_est'][i],
        'AME_SpatialEffect2': spatialeffect2_payload['AME_est']['taud_est'][i],
        'diff': float(spatialeffect2_payload['AME_est']['taud_est'][i]) - float(spatialeffect_payload['AME_est']['taud_est'][i])
    })


{'d': 0, 'AME_SpatialEffect': -0.010819833222700805, 'AME_SpatialEffect2': -0.010819833222700805, 'diff': 0.0}
{'d': 500, 'AME_SpatialEffect': -0.004246608025969436, 'AME_SpatialEffect2': -0.004246608025969436, 'diff': 0.0}
{'d': 1000, 'AME_SpatialEffect': -0.010741847850956741, 'AME_SpatialEffect2': -0.010741847850956741, 'diff': 0.0}
{'d': 1500, 'AME_SpatialEffect': -0.005786193853801681, 'AME_SpatialEffect2': -0.005786193853801681, 'diff': 0.0}
{'d': 2000, 'AME_SpatialEffect': -0.007306674457442727, 'AME_SpatialEffect2': -0.007306674457442727, 'diff': 0.0}
{'d': 2500, 'AME_SpatialEffect': -0.007534918024839169, 'AME_SpatialEffect2': -0.007534918024839169, 'diff': 0.0}
{'d': 3000, 'AME_SpatialEffect': -0.008276730344906951, 'AME_SpatialEffect2': -0.008276730344906951, 'diff': 0.0}
{'d': 3500, 'AME_SpatialEffect': -0.00824554322265277, 'AME_SpatialEffect2': -0.00824554322265277, 'diff': 0.0}
{'d': 4000, 'AME_SpatialEffect': -0.005211255781442843, 'AME_SpatialEffect2': -0.0052112557814